In [ ]:
#Necessary Dependencies 
import sys
sys.path.append("..")

import importlib
import tensorflow as tf
import DataModules.DataAugmentation
importlib.reload(DataModules.DataAugmentation)

from tensorflow import keras
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Maximum, Input
from DataModules.DataAugmentation import get_datasets

In [ ]:
#Loading Data

train_dataset1, test_dataset1, train_dataset2, test_dataset2, train_dataset3, test_dataset3, train_dataset4, test_dataset4, data_augmentation = get_datasets()

combined_train = train_dataset1.concatenate(train_dataset2)
combined_train = combined_train.concatenate(train_dataset3)
combined_train = combined_train.concatenate(train_dataset4)

combined_test = test_dataset1.concatenate(test_dataset2)
combined_test = combined_test.concatenate(test_dataset3)
combined_test = combined_test.concatenate(test_dataset4)

In [ ]:
#LOADING THE NEEDED MODELS

GoogleNet_RNFL(TM) = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/GoogleNet/googlenet(rnfl_tm).h5')
ResNet_GCIPL(DM) = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/ResNet/resnet(gcipl_dm).h5')
NASNet_GCIPL(TM) = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/NASnet/nasnet(gcipl_tm).h5')
DenseNet_RNFL(DM) = load_model('/content/drive/MyDrive/Colab Notebooks/Main Project/Saved Models/Densenet/densenet_new(rnfl_dm).h5')

In [ ]:
#Combined Model

models = [GoogleNet_RNFL(TM), ResNet_GCIPL(DM), NASNet_GCIPL(TM), DenseNet_RNFL(DM)]
model_input = Input(shape=(224, 224, 3))
model_output = [model(model_input) for model in models]
ensemble_output = Maximum()(model_output)

ensemble_1 = Model(inputs=model_input, outputs=ensemble_output)

In [ ]:
ensemble_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
    )

history = {}

history = ensemble_1.fit(
        combined_train,
        validation_data=combined_test,
        epochs=150
    )